# Query a Database For Updates and Add to an Existing Dataset
Demonstrate how to get ingest incremental updates from a SQL database into an existing Rhino dataset

#### Prerequisites
1. Have a SQL DB that is open to connections from a Rhino Client, with access credentials for read-only access to this DB
2. Have a project where you are either part of the project's lead workgroup and the DB is within your site, or where there is a collaborator in the project that has the DB at their site
3. Ensure the required site-level permissions for SQL querying (Import and export datasets, View dataset analytics) are enabled for the site that has the DB
4. Have an exising Rhino dataset with a matching schema as the query you wish to run.

### 1. Initialization and Login

In [1]:
from getpass import getpass
from pprint import pprint

import rhino_health as rh
from rhino_health.lib.endpoints.sql_query.sql_query_dataclass import (
    SQLQueryImportInput,
    SQLQueryInput,
    SQLServerTypes,
    ConnectionDetails,
)
from rhino_health.lib.metrics import Count, FilterType, Mean, StandardDeviation

In [5]:
my_username = "dan@rhinohealth.com" # Replace this with the email you use to log into Rhino Health
ecr_base_uri = "XXXXXXXXXXXX.dkr.ecr.us-east-1.amazonaws.com" # Replace XXXXXXXXXXXX with the UID of your project

print("Logging In")
session = rh.login(username=my_username, password=getpass())
print("Logged In")

Logging In
Exception: No active account found with the given credentials


### 2. Connection Setup

When specifying the connection details, ensure that you provide the server_type using the approved SQLServerTypes enum. This step ensures that your server is supported and compatible with the querying process.

In [ ]:
sql_db_user = "" # Replace this with your DB username (make sure the user has read-only permissions to the DB).
sql_db_password = "" # Replace this with your DB password. For better security, consider using an environment variable, e.g. - os.getenv("DB_PASSWORD")., or using getpass.getpass() to type in the password
external_server_url = "" # Replace this with url + port of the SQL DB you want to query (ie "{url}:{port}").
db_name = "" # Replace this with your DB name.

connection_details = ConnectionDetails(
    server_user=sql_db_user,
    password=sql_db_password,
    server_type=SQLServerTypes.POSTGRESQL, # Replace POSTGRESQL with the relevant type of your sql server (See docs for all supported types).
    server_url=external_server_url,
    db_name=db_name
)

#### Collect all necessary parameter for queries:

Note - You can copy your project UID from the projects page in the Rhino Health dashboard (GUI) under the 3-dot menu

In [ ]:
project_uid = session.project.get_project_by_name('Your project name').uid
# Set the workgroup_uid to the workgroup the DB belongs to.
workgroup_uid = session.project.get_collaborating_workgroups(project_uid)[0].uid